In [ ]:
from run.base import OptimizerFactory

/opt/anaconda3/envs/mlmcbo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
optimizer = BaseExperiment()

In [ ]:
import numpy
numpy.__version__

'1.26.4'

In [37]:
import numpy as np
import torch
from torch import Tensor

from botorch.fit import fit_gpytorch_mll
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.optim import optimize_acqf
from botorch.test_functions import Hartmann
from botorch.acquisition.analytic import ExpectedImprovement
from botorch.acquisition.monte_carlo import qExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler

from gpytorch.mlls import ExactMarginalLogLikelihood
from mlmc import ExpectedImprovementTwoStepLookahead

torch.manual_seed(0)
torch.set_default_dtype(torch.double)

In [29]:
DIM = 6
N_INIT = 10
N_ITER = 20

REFIT_ON_UPDATE = True      # like Ax's refit_on_update
WARM_START_REFIT = True     # like Ax's warm_start_refit
BATCH_SIZE = 4

blackbox = Hartmann(dim=DIM, negate=True)

bounds = torch.stack(
    [torch.zeros(DIM, dtype=torch.double), torch.ones(DIM, dtype=torch.double)]
)

In [26]:
blackbox.bounds

tensor([[0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1.]], dtype=torch.float32)

In [25]:
blackbox(blackbox.optimizers).item()

3.3223679065704346

In [18]:
blackbox.optimal_value

3.32237

In [27]:

def generate_initial_data(n=N_INIT):
    """Random initial design + evaluations."""
    train_X = torch.rand(n, DIM, dtype=torch.double)
    train_Y = blackbox(train_X).unsqueeze(-1)  # (n, 1)
    return train_X, train_Y

def build_model(train_X: Tensor, train_Y: Tensor):
    """Create a new SingleTaskGP + MLL on the given data."""
    model = SingleTaskGP(
        train_X,
        train_Y,
        input_transform=Normalize(d=DIM),
        outcome_transform=Standardize(m=1),
    )
    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    return model, mll


def fit_model(
    train_X: Tensor,
    train_Y: Tensor,
    state_dict: dict | None = None,
) -> tuple[SingleTaskGP, ExactMarginalLogLikelihood, dict]:
    """
    Build and fit a GP model on (train_X, train_Y).

    - If `state_dict` is provided, we warm-start the new model from it
      (mimicking Ax's warm_start_refit).
    - Then we fit hyperparameters (refit_on_update=True behavior).
    """
    model, mll = build_model(train_X, train_Y)

    if state_dict is not None and WARM_START_REFIT:
        # Load previous hyperparameters into the new model
        model.load_state_dict(state_dict)

    if REFIT_ON_UPDATE:
        fit_gpytorch_mll(mll)

    # Save updated hyperparameters to warm start next iteration
    new_state_dict = model.state_dict()
    return model, mll, new_state_dict


In [ ]:
def gen_batch_candidates(
    model: SingleTaskGP,
    best_f: float,
    q: int = BATCH_SIZE,
) -> tuple[Tensor, Tensor]:
    """
    Generate a batch of q candidates using qEI (MC).
    Returns:
      new_x: (q, d)
      new_y: (q, 1)
    """
    # sampler = SobolQMCNormalSampler(num_samples=torch.Size([128]))

    acqf = qExpectedImprovement(
        model=model,
        best_f=best_f,
        # sampler=sampler,
    )

    candidates, _ = optimize_acqf(
        acq_function=acqf,
        bounds=bounds,
        q=q,
        num_restarts=10,
        raw_samples=128,
        # optional but often helpful:
        # options={"batch_limit": 5, "maxiter": 200},
    )

    new_x = candidates.detach()                 # (q, d)
    new_y = neg_hartmann6(new_x).unsqueeze(-1) # (q, 1)
    return new_x, new_y

def main():
    # Initial data
    train_X, train_Y = generate_initial_data(N_INIT)

    # Initial model fit (no warm start yet)
    model, mll, state_dict = fit_model(train_X, train_Y, state_dict=None)

    for i in range(N_ITER):
        # Generate new candidate
        best_f = train_Y.max().item()
        # new_x, new_y = gen_one_candidate(model, best_f)
        new_x, new_y = gen_batch_candidates(model, best_f, q=4)

        # Attach data
        train_X = torch.cat([train_X, new_x], dim=0)
        train_Y = torch.cat([train_Y, new_y], dim=0)

        # Rebuild + warm-start + refit on ALL data
        model, mll, state_dict = fit_model(train_X, train_Y, state_dict=state_dict)

        print(f"Iter {i+1:02d} | Observed best value: {train_Y.max().item():.4f}")

    best_value, best_idx = train_Y.max(dim=0)
    best_point = train_X[best_idx]

    print("\nBest point found:", best_point)
    print("Best objective value (neg Hartmann6):", best_value.item())

main()

Iter 01 | Observed best value: 0.5605
Iter 02 | Observed best value: 1.0710


/opt/anaconda3/envs/mlmcbo/lib/python3.11/site-packages/botorch/optim/initializers.py:403: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  warnings.warn(
/opt/anaconda3/envs/mlmcbo/lib/python3.11/site-packages/botorch/optim/initializers.py:403: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  warnings.warn(


Iter 03 | Observed best value: 1.0710
Iter 04 | Observed best value: 1.1417


/opt/anaconda3/envs/mlmcbo/lib/python3.11/site-packages/botorch/optim/initializers.py:403: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  warnings.warn(
/opt/anaconda3/envs/mlmcbo/lib/python3.11/site-packages/botorch/optim/initializers.py:403: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  warnings.warn(


Iter 05 | Observed best value: 1.1417
Iter 06 | Observed best value: 1.1417


...

/opt/anaconda3/envs/mlmcbo/lib/python3.11/site-packages/botorch/optim/initializers.py:403: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  warnings.warn(


Iter 17 | Observed best value: 1.5262
Iter 18 | Observed best value: 1.5262


/opt/anaconda3/envs/mlmcbo/lib/python3.11/site-packages/botorch/optim/initializers.py:403: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  warnings.warn(
/opt/anaconda3/envs/mlmcbo/lib/python3.11/site-packages/botorch/optim/initializers.py:403: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  warnings.warn(


Iter 19 | Observed best value: 1.5262
Iter 20 | Observed best value: 1.5262

Best point found: tensor([[0.3861, 0.3448, 0.7692, 0.0694, 0.2514, 0.8140]])
Best objective value (neg Hartmann6): 1.5261873154669663


/opt/anaconda3/envs/mlmcbo/lib/python3.11/site-packages/botorch/optim/initializers.py:403: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  warnings.warn(


In [28]:
def gen_one_candidate(model: SingleTaskGP, best_f: float) -> tuple[Tensor, Tensor]:
    """Generate one new candidate via EI and evaluate it."""
    acqf = ExpectedImprovement(model=model, best_f=best_f)

    candidate, _ = optimize_acqf(
        acq_function=acqf,
        bounds=blackbox.bounds,
        q=1,
        num_restarts=10,
        raw_samples=64,
    )

    new_x = candidate.detach()
    new_y = blackbox(new_x).unsqueeze(-1)
    return new_x, new_y

def main():
    # Initial data
    train_X, train_Y = generate_initial_data(N_INIT)

    # Initial model fit (no warm start yet)
    model, mll, state_dict = fit_model(train_X, train_Y, state_dict=None)

    for i in range(N_ITER):
        # Generate new candidate
        best_f = train_Y.max().item()
        new_x, new_y = gen_one_candidate(model, best_f)

        # Attach data
        train_X = torch.cat([train_X, new_x], dim=0)
        train_Y = torch.cat([train_Y, new_y], dim=0)

        # Rebuild + warm-start + refit on ALL data (Ax-style)
        model, mll, state_dict = fit_model(train_X, train_Y, state_dict=state_dict)

        print(f"Iter {i+1:02d} | Observed best value: {train_Y.max().item():.4f}")

    best_value, best_idx = train_Y.max(dim=0)
    best_point = train_X[best_idx]

    print("\nBest point found:", best_point)
    print("Best objective value (neg Hartmann6):", best_value.item())


main()


Iter 01 | Observed best value: 1.1875
Iter 02 | Observed best value: 1.1877
Iter 03 | Observed best value: 1.1877
Iter 04 | Observed best value: 1.1877
Iter 05 | Observed best value: 2.1128
Iter 06 | Observed best value: 2.1128
Iter 07 | Observed best value: 2.1128
Iter 08 | Observed best value: 2.2979
Iter 09 | Observed best value: 2.2979
Iter 10 | Observed best value: 2.2979
Iter 11 | Observed best value: 2.2979
Iter 12 | Observed best value: 2.2979
Iter 13 | Observed best value: 2.2979
Iter 14 | Observed best value: 2.2979
Iter 15 | Observed best value: 2.2979
Iter 16 | Observed best value: 2.2979
Iter 17 | Observed best value: 2.2979
Iter 18 | Observed best value: 2.2979
Iter 19 | Observed best value: 2.2979
Iter 20 | Observed best value: 2.2979

Best point found: tensor([[0.4391, 0.9089, 0.1889, 0.5855, 0.6071, 0.1739]])
Best objective value (neg Hartmann6): 2.2979328550562075
